Join with historical data on religion in countries

In [1]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
FILE_PATH_IN_1 = "avibert/diffusion/World religions 1945-2000.SAV"
FILE_PATH_IN_2 = "avibert/diffusion/df_trips_countries.parquet"

with fs.open(FILE_PATH_IN_1, "rb") as file_in:
    df_religions = pd.read_spss(file_in)

with fs.open(FILE_PATH_IN_2, "rb") as file_in_2:
    df_trips = pd.read_parquet(file_in_2)


In [2]:
df_religions = df_religions[["YEAR","ISO3",'CHPRTPCT', 'CHCATPCT', 'CHORTPCT', 'CHANGPCT', 'CHOTHPCT', 'CHGENPCT',
       'JDORPCT', 'JDCNPCT', 'JDRFPCT', 'JDOTPCT', 'JDGENPCT', 'ISSUNPCT',
       'ISSHIPCT', 'ISIBDPCT', 'ISNATPCT', 'ISALAPCT', 'ISAHMPCT', 'ISLOTPCT',
       'ISGENPCT', 'BUMAHPCT', 'BUTHRPCT', 'BUOTHPCT', 'BUGENPCT', 'ZOGENPCT',
       'HIGENPCT', 'SIGENPCT', 'SHGENPCT', 'BAGENPCT', 'TAGENPCT', 'JAGENPCT',
       'COGENPCT', 'SYGENPCT', 'ANGENPCT', 'NORELPCT', 'OTGENPCT']]

In [3]:
df_religions["YEAR"] = pd.to_numeric(df_religions["YEAR"].astype(str).str.replace(",",""))
df_religions["Majority"] = df_religions.drop(columns=["YEAR","ISO3"]).idxmax(axis=1)

df_joint = pd.merge_asof(
    df_trips.sort_values("Year"),
    df_religions.sort_values("YEAR"),
    left_on = "Year",
    right_on = "YEAR",
    by = "ISO3",
    direction = "nearest"
)

In [4]:
url = "https://www.pewresearch.org/religion/2011/12/19/table-christian-population-as-percentages-of-total-population-by-country/"
pew = pd.read_html(url)[0]

ImportError: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.

In [ ]:
df_joint["CHCATPCT"].describe()

In [ ]:
df_joint.loc[:,["Pope","CHCATPCT"]].groupby("Pope").describe()

In [ ]:
df_joint.loc[:,["Pope","Majority"]].groupby("Pope").agg()